In [1]:
# Load all required packages
import concurrent.futures
import os
import pandas as pd
from tqdm import tqdm
import scanpy as sc
if not os.path.exists('../../../data/rna/plasma/outputs'):
    os.makedirs('../../../data/rna/plasma/outputs')
import concurrent.futures
import pyarrow as pa
import fastparquet

### Read in objects and Inspect
- Make sure raw counts are accessible
- Filter to remove mitochondrial, ribosomal, hb, Ig genes
- Add in tumor cluster labels as NDMM_Cluster#

In [6]:
# download processed BMMC plasma subset files from allenimmunology.org
hp.cache_files(["697722a7-35cc-4f11-8265-ecccc93358be"])

['/home/workspace/input/849499887/FH_MM_Year4/697722a7-35cc-4f11-8265-ecccc93358be/dhBEZegaUf/bmmc-plasma-processed.h5ad']

In [8]:
# read in anndata object that has raw counts saved in adata.layers
plasma = sc.read_h5ad('../../../data/rna/bmmc-plasma-processed.h5ad')

In [8]:
# filter for only pretx and healthy
plasma_filtered = plasma[
    plasma.obs['sample.visitDetails'].isin(['MM Pre-Treatment', 'Healthy'])
].copy()

In [9]:
# sanity check
plasma_filtered.obs['sample.visitDetails'].unique()

['Healthy', 'MM Pre-Treatment']
Categories (2, object): ['Healthy', 'MM Pre-Treatment']

In [10]:
# check that data includes raw counts for pseudobulk
plasma_filtered.raw.X.min(), plasma_filtered.raw.X.max(), type(plasma_filtered.raw.X)

(np.uint32(0), np.uint32(78954), scipy.sparse._csr.csr_matrix)

In [11]:
# remove ig genes before aggregating counts
pretx_plasma = plasma_filtered.raw.to_adata()

mt_mask = pretx_plasma.var_names.str.startswith("MT-")
ribo_mask = pretx_plasma.var_names.str.startswith(("RPS", "RPL"))
hb_mask = pretx_plasma.var_names.str.contains("^HB[^(P)]")
ig_mask = pretx_plasma.var_names.str.startswith(('IGK', 'IGL', 'IGH'))

mask = mt_mask | ribo_mask | hb_mask| ig_mask
pretx_plasma = pretx_plasma[:, ~mask]

pretx_plasma.raw = pretx_plasma

In [12]:
# re-name
plasma_filtered = pretx_plasma

In [13]:
# get orginal tumor cluster labels 
df = pd.read_parquet(
    '../../../data/rna/infercnv/metadata/plasma_pretx_state_annotation.parquet',
    engine='fastparquet'

)

In [14]:
# check metadata
df

,plasma_annotation_leiden,plasma_labels_l3
barcodes,,
1d4b50cae88711eca2fc4e28d07d5108,0,MS-OxPhos
1d562388e88711eca2fc4e28d07d5108,0,MS-OxPhos
1d5c134ce88711eca2fc4e28d07d5108,7,Myeloid-Evasive
1d5d79e4e88711eca2fc4e28d07d5108,3,CD2-ISG+
1d63d1fee88711eca2fc4e28d07d5108,7,Myeloid-Evasive
...,...,...
06784978e8a711ec9cc02a56aa421d9a,1,HY-Metabolic
06785e54e8a711ec9cc02a56aa421d9a,2,MS-ImmuneLow
0678d9d8e8a711ec9cc02a56aa421d9a,0,MS-OxPhos


In [15]:
# ensure parquet index is named barcodes
df.index.name = 'barcodes'

# make sure plasma obs also uses barcodes
plasma_filtered.obs.index.name = 'barcodes'

# align by index and add metadata columns
plasma_filtered.obs = plasma_filtered.obs.join(df, how='left')

In [16]:
# check
plasma_filtered.obs['plasma_labels_l3'].unique()

[NaN, 'MS-OxPhos', 'Myeloid-Evasive', 'CD2-ISG+', 'HY-Metabolic', ..., 'PR-CellCycleHigh', 'HY-ISGhigh', 'CD1-DevHh', 'HY-CheckpointLow', 'MS-ProteasomeStress']
Length: 13
Categories (12, object): ['MS-OxPhos', 'HY-Metabolic', 'MS-ImmuneLow', 'CD2-ISG+', ..., 'PR-CellCycleHigh', 'CD1-DevHh', 'HY-CheckpointLow', 'MS-ProteasomeStress']

In [17]:
# add 'healthy_plasma' to the categories
plasma_filtered.obs['plasma_labels_l3'] = plasma_filtered.obs['plasma_labels_l3'].cat.add_categories('Healthy Plasma')

# fill NaN
plasma_filtered.obs['plasma_labels_l3'] = plasma_filtered.obs['plasma_labels_l3'].fillna('Healthy Plasma')

In [18]:
# check leiden labels
plasma_filtered.obs['plasma_annotation_leiden']

barcodes
88fea8222a1f11efb3fd9a4aca01706b    NaN
8905c3a02a1f11efb3fd9a4aca01706b    NaN
89088b942a1f11efb3fd9a4aca01706b    NaN
8927faa62a1f11efb3fd9a4aca01706b    NaN
ed8d613c2a1211efa7794ab4e1f01ac2    NaN
                                   ... 
5a3783742a0611efbe9a1a408edca6c0    NaN
5a3e7f762a0611efbe9a1a408edca6c0    NaN
5a4021aa2a0611efbe9a1a408edca6c0    NaN
5a420b642a0611efbe9a1a408edca6c0    NaN
5a44adb02a0611efbe9a1a408edca6c0    NaN
Name: plasma_annotation_leiden, Length: 55517, dtype: category
Categories (12, object): ['0', '1', '2', '3', ..., '8', '9', '10', '11']

In [19]:
# add "Healthy Plasma" to the categories
plasma_filtered.obs['plasma_annotation_leiden'] = plasma_filtered.obs['plasma_annotation_leiden'].cat.add_categories('Healthy Plasma')

# fill NaNs
plasma_filtered.obs['plasma_annotation_leiden'] = plasma_filtered.obs['plasma_annotation_leiden'].fillna('Healthy Plasma')

In [20]:
# re-name tumor cluster labels to new naming convention NDMM_cluster#
plasma_filtered.obs['tumor_cluster_labels'] = plasma_filtered.obs['plasma_annotation_leiden'].apply(
    lambda x: x if x == 'Healthy Plasma' else f'NDMM_{x}'
)

In [21]:
# sanity check
plasma_filtered.obs['tumor_cluster_labels'].unique()

['Healthy Plasma', 'NDMM_0', 'NDMM_7', 'NDMM_3', 'NDMM_1', ..., 'NDMM_8', 'NDMM_4', 'NDMM_9', 'NDMM_10', 'NDMM_11']
Length: 13
Categories (13, object): ['NDMM_0', 'NDMM_1', 'NDMM_2', 'NDMM_3', ..., 'NDMM_9', 'NDMM_10', 'NDMM_11', 'Healthy Plasma']

In [22]:
# Use the raw counts from plasma_filtered
X_all = plasma_filtered.raw.X
plasma_cells = plasma_filtered  # name change
sample_ids = plasma_cells.obs['sample.sampleKitGuid'].unique()

for sample_id in sample_ids:
    # select cells for this kit ID
    sample_cells = plasma_cells[plasma_cells.obs['sample.sampleKitGuid'] == sample_id]
    
    # get unique tumor clusters in this sample
    clusters = sample_cells.obs['tumor_cluster_labels'].unique()
    
    cluster_counts = {}
    
    for cluster in clusters:
        # select cells for this cluster
        cluster_cells = sample_cells[sample_cells.obs['tumor_cluster_labels'] == cluster]
        
        # get indices in the original plasma cell object
        idx = cluster_cells.obs_names.map(lambda x: plasma_cells.obs_names.get_loc(x))
        
        # sum raw counts across all cells in this cluster
        if hasattr(X_all, "toarray"):
            counts = X_all[idx, :].toarray().sum(axis=0)
        else:
            counts = X_all[idx, :].sum(axis=0)
        
        cluster_counts[cluster] = counts

    # use raw.var_names as gene index
    index = plasma_cells.raw.var_names

    # build df columns = clusters, rows = genes
    df = pd.DataFrame(cluster_counts, index=index)
    df.index.name = 'Gene'
    
    # save pseudo-bulk CSV for each Kit ID
    out_file = f"../../../data/rna/plasma/outputs/pseudo_bulk_matrices_raw_counts/{sample_id}_pseudo_bulk.csv"
    df.to_csv(out_file)
    print(f"✅ Saved pseudo-bulk for sample {sample_id} → {out_file}")

✅ Saved pseudo-bulk for sample KT01671 → outputs/pseudo_bulk_matrices_raw_counts/KT01671_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02437 → outputs/pseudo_bulk_matrices_raw_counts/KT02437_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02424 → outputs/pseudo_bulk_matrices_raw_counts/KT02424_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02421 → outputs/pseudo_bulk_matrices_raw_counts/KT02421_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02420 → outputs/pseudo_bulk_matrices_raw_counts/KT02420_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02427 → outputs/pseudo_bulk_matrices_raw_counts/KT02427_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02434 → outputs/pseudo_bulk_matrices_raw_counts/KT02434_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02432 → outputs/pseudo_bulk_matrices_raw_counts/KT02432_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02430 → outputs/pseudo_bulk_matrices_raw_counts/KT02430_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02419 → outputs/pseudo

In [15]:
# make an aggregate file for deseq2 input
# directory with pseudo-bulk CSVs
pb_dir = Path("../../../data/rna/plasma/outputs/pseudo_bulk_matrices_raw_counts/")
pb_files = list(pb_dir.glob("*_pseudo_bulk.csv"))

counts_list = []
metadata_list = []

for f in pb_files:
    sample_id = f.stem.replace("_pseudo_bulk", "")
    df = pd.read_csv(f, index_col=0)  # index = genes
    
    # columns = tumor clusters + Healthy Plasma
    for col in df.columns:
        col_name = f"{sample_id}_{col}"
        counts_list.append(df[col].rename(col_name))
        
        # get metadata ie sample_id, cluster label, condition
        condition = "Healthy Plasma" if "Healthy Plasma" in col else "NDMM"
        metadata_list.append({"colname": col_name, "sample_id": sample_id, "cluster": col, "condition": condition})

# combine counts
counts_matrix = pd.concat(counts_list, axis=1)

# create metadata dataframe
colData = pd.DataFrame(metadata_list).set_index("colname")

# save outputs for input into deseq2
counts_matrix.to_csv("../../../data/rna/pseudobulk/outputs/pseudo_bulk_matrices_raw_counts/combined_pseudobulk_counts.csv")
colData.to_csv("../../../data/rna/pseudobulk/outputs/pseudo_bulk_matrices_raw_counts/combined_pseudobulk_colData.csv")

### Get log-normalized pseudobulk for making dotplots

In [ ]:
# read in the normalized data
adata = sc.read_h5ad('../../../data/rna/ndmm_tumor_clusters_new_label_20251023.h5ad')
adata = adata[adata.obs['source'] != 'Healthy External Atlas'].copy()

In [31]:
# Use the log-normalized counts from adata
X_all = adata.X
sample_ids = adata.obs['sample.sampleKitGuid'].unique()

# make sure output directory exists
os.makedirs("../../../data/rna/plasma/outputs/pseudo_bulk_matrices_log_counts", exist_ok=True)

for sample_id in sample_ids:
    # select cells for this kit id
    sample_cells = adata[adata.obs['sample.sampleKitGuid'] == sample_id]
    
    # get unique tumor clusters in this sample
    clusters = sample_cells.obs['tumor_cluster_labels'].unique()
    
    cluster_counts = {}
    
    for cluster in clusters:
        # select cells for this cluster
        cluster_cells = sample_cells[sample_cells.obs['tumor_cluster_labels'] == cluster]
        
        # get indices in the original object
        idx = cluster_cells.obs_names.map(lambda x: adata.obs_names.get_loc(x))
        
        # sum log-normalized counts across all cells in this cluster
        if hasattr(X_all, "toarray"):
            counts = X_all[idx, :].toarray().sum(axis=0)
        else:
            counts = X_all[idx, :].sum(axis=0)
        
        cluster_counts[cluster] = counts

    # use var_names as gene index
    index = adata.var_names  # log-normalized counts are in .X, so just var_names here
    
    # make df columns = clusters, rows = genes
    df = pd.DataFrame(cluster_counts, index=index)
    df.index.name = 'Gene'
    
    # save pseudo-bulk CSV for each kit ID
    out_file = f"../../../data/rna/plasma/outputs/pseudo_bulk_matrices_log_counts/{sample_id}_pseudo_bulk.csv"
    df.to_csv(out_file)
    print(f"✅ Saved pseudo-bulk for sample {sample_id} → {out_file}")

✅ Saved pseudo-bulk for sample KT02437 → outputs/pseudo_bulk_matrices_log_counts/KT02437_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02424 → outputs/pseudo_bulk_matrices_log_counts/KT02424_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02421 → outputs/pseudo_bulk_matrices_log_counts/KT02421_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02420 → outputs/pseudo_bulk_matrices_log_counts/KT02420_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02427 → outputs/pseudo_bulk_matrices_log_counts/KT02427_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02434 → outputs/pseudo_bulk_matrices_log_counts/KT02434_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02432 → outputs/pseudo_bulk_matrices_log_counts/KT02432_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02430 → outputs/pseudo_bulk_matrices_log_counts/KT02430_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02419 → outputs/pseudo_bulk_matrices_log_counts/KT02419_pseudo_bulk.csv
✅ Saved pseudo-bulk for sample KT02436 → outputs/pseudo

In [9]:
# make aggregated file from the log normalized pseudobulk
in_dir = "../../../data/rna/plasma/outputs/pseudo_bulk_matrices_log_counts_downsampled/" 
out_file = os.path.join(in_dir, "KT_aggregated_mean.csv") 

# get files that start with Kit ID
files = sorted(glob.glob(os.path.join(in_dir, "KT*.csv")))

if not files:
    raise FileNotFoundError("No files starting with 'KT' found in the output directory.")

# read in all files
all_data = []
for f in files:
    df = pd.read_csv(f, index_col=0)  # rows = genes, columns = clusters
    df['patient'] = os.path.basename(f).split('_')[0]  # extract KT ID (e.g. KT123)
    all_data.append(df)


# combine in long
df_long = pd.concat(all_data, axis=0, keys=[os.path.basename(f).split('_')[0] for f in files])
df_long = df_long.reset_index(level=0, drop=True)
df_long = df_long.reset_index(names='Gene')

# tidy
df_melted = df_long.melt(id_vars=['Gene', 'patient'], var_name='cluster', value_name='expr')

# aggregate (mean expression)
df_mean = (
    df_melted
    .groupby(['Gene', 'cluster'], as_index=False)['expr']
    .mean()
    .pivot(index='Gene', columns='cluster', values='expr')
    .sort_index(axis=1)
)

# save csv
df_mean.to_csv(out_file)
print(f"✅ Aggregated pseudobulk matrix saved → {out_file}")

✅ Aggregated pseudobulk matrix saved → outputs/pseudo_bulk_matrices_log_counts/KT_aggregated_mean.csv
